### Section 4: Final Prediction Pipeline


In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
TEST_DATA_PATH = "test_data.csv"
SUBMISSION_PATH = "Group10_Generalization.csv"

# Models to load (only the ones we trained)
# Clusters 0 and 5 are "Constant 0" and have no files.
MODEL_FILES = {
    1: "cluster1_stacking.joblib",
    2: "cluster2_stacking.joblib",
    3: "cluster3_stacking.joblib",
    4: "cluster4_stacking.joblib"
}

# Load Preprocessing & Cluster Predictor
cluster_pkg = joblib.load("cluster_id_predictor.joblib")
scaler = cluster_pkg["scaler"]
cluster_model = cluster_pkg["model"]
feature_names = cluster_pkg["features"]

print("Loaded Cluster Predictor and Scaler.")

# Load Subgroup Models
subgroup_models = {}
for c_id, filename in MODEL_FILES.items():
    if os.path.exists(filename):
        print(f"Loading model for Cluster {c_id}...")
        pkg = joblib.load(filename)
        subgroup_models[c_id] = pkg["model"]
    else:
        print(f"WARNING: Model file {filename} not found! Predictions for Cluster {c_id} will fail.")

Loaded Cluster Predictor and Scaler.
Loading model for Cluster 1...
Loading model for Cluster 2...
Loading model for Cluster 3...
Loading model for Cluster 4...


In [3]:
df_test = pd.read_csv(TEST_DATA_PATH)
print(f"\nLoaded Test Data. Shape: {df_test.shape}")

# Store Index for submission
test_indices = df_test["Index"]

# 1. Preprocess: Select features and Scale
# We must use EXACTLY the same features used to train the cluster predictor
X_test_raw = df_test[feature_names]

X_test_scaled = scaler.transform(X_test_raw)

# 2. Predict Cluster ID for each test company
print("Predicting Cluster IDs for test data...")
test_clusters = cluster_model.predict(X_test_scaled)


Loaded Test Data. Shape: (1012, 96)
Predicting Cluster IDs for test data...


In [4]:
# ==========================================
# 4. Generate Final Predictions
# ==========================================
final_predictions = []
debug_counts = {i: 0 for i in range(6)} # Track how many companies go to each cluster

print("\nRouting test companies to subgroup models...")

for i in range(len(df_test)):
    c_id = test_clusters[i]
    row_index = test_indices.iloc[i]
    
    # Track stats
    debug_counts[c_id] += 1
    
    # Logic:
    # If Cluster 0 or 5 -> Constant 0 (Safe)
    # If Cluster 1, 2, 3, 4 -> Use loaded Stacking Model
    
    if c_id in [0, 5]:
        pred = 0 # Constant prediction
    elif c_id in subgroup_models:
        # We need to extract the specific features this model expects
        # (Though in our code, all models largely used the same 'top_features', 
        # it's safer to pass the whole row subset if models align)
        
        # Note: The stacking models expect the specific feature subset used during training.
        # Since we standardized 'features_to_use' in all notebooks to match 'top_features_for_clustering',
        # we can assume X_test_raw (subsetted) is close, BUT sklearn models need exact shape.
        
        # Safer approach: The models were trained on 'features_to_use' loaded from the common joblib.
        # So X_test_raw matches that feature set exactly.
        
        # We need to pass a 2D array (1 sample)
        sample = X_test_raw.iloc[[i]]
        pred = subgroup_models[c_id].predict(sample)[0]
    else:
        # Fallback (Should not happen if files exist)
        print(f"Warning: No model for Cluster {c_id}. Defaulting to 0.")
        pred = 0
        
    final_predictions.append(pred)


Routing test companies to subgroup models...


In [5]:
submission_df = pd.DataFrame({
    "Index": test_indices,
    "Bankrupt?": final_predictions
})

submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f"Total Predictions: {len(submission_df)}")
print(f"Predicted Bankruptcies: {sum(final_predictions)}")
print("\nCluster Distribution in Test Set:")
for c_id, count in sorted(debug_counts.items()):
    print(f"  Cluster {c_id}: {count} companies")

# Sanity Check
pct_bankrupt = (sum(final_predictions) / len(submission_df)) * 100
print(f"\nFinal Predicted Bankruptcy Rate: {pct_bankrupt:.2f}%")
if pct_bankrupt > 20:
    print("Prediction rate is > 20%. You might be overfitting.")
else:
    print("Prediction rate is within expected bounds.")

Total Predictions: 1012
Predicted Bankruptcies: 103

Cluster Distribution in Test Set:
  Cluster 0: 86 companies
  Cluster 1: 346 companies
  Cluster 2: 460 companies
  Cluster 3: 0 companies
  Cluster 4: 120 companies
  Cluster 5: 0 companies

Final Predicted Bankruptcy Rate: 10.18%
Prediction rate is within expected bounds.
